# 12 — RQ2: gom cụm review actionable bằng BERTopic, loại chủ đạo và type purity (Kaggle hoặc máy local)

**Đầu vào**

`actionable_reviews.csv` của notebook 11: mọi review của corpus Shopee mà cascade ViSoBERT (amendment v2.2) gọi ACTIONABLE, kèm nhãn
tầng 2 (BUG / FEATURE / UX).

**Gom chung rồi gắn loại**

Toàn bộ review actionable được gom trong một lần chạy (CLAUDE.md §1). Nhãn BUG / FEATURE / UX không dùng để chia dữ liệu trước khi
gom; nó được gắn vào cụm sau khi gom để xác định loại chủ đạo. Gom riêng từng loại là ablation tùy chọn, không làm ở đây.

**Quy trình (research_protocol, mục RQ2)**

Sentence embedding của văn bản gốc → UMAP → HDBSCAN → c-TF-IDF trên văn bản đã tách từ. `min_topic_size` chọn trên lưới
{15, 20, 30, 50, 75} theo coherence C_v (Q12); cấu hình suy biến (một cụm chiếm quá 30% review đã gom, hoặc quá 100 cụm) bị loại.
Cấu hình thắng được giảm outlier và gộp các cụm gần trùng. Loại chủ đạo: một loại chiếm ít nhất 60% review của cụm, nếu không thì
`mixed` kèm cờ cho BA (Q14).

**Cấu hình**

Mọi tham số đọc từ `03_configs/dev/rq2_topics_dev_config.yaml`; sau khi duyệt cụm, cấu hình thắng được khóa sang `03_configs/frozen/`.

**Nơi chạy**

Trên Kaggle: mã nguồn được clone từ repository như notebook 11, thư viện gom cụm được cài ở cell đầu, đầu vào là output của notebook 11
(gắn làm input), kết quả ghi vào `/kaggle/working/rq2_topics/` để tải về `05_experiments/rq2_topics/`. Trên máy local: đầu vào và kết
quả nằm trong `05_experiments/rq2_topics/`. Đặt tên cụm bằng LLM chỉ chạy trên máy local, vì API key đọc từ `.env`.

In [ ]:
import os, sys, json, datetime, subprocess, warnings
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
    ROOT = '/kaggle/working/repo'
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not os.path.exists(ROOT):
        r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', ROOT], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
    subprocess.run(['git', '-C', ROOT, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
    del token
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bertopic==0.17.4', 'underthesea==9.5.0', 'umap-learn',
                    'hdbscan', 'gensim'], check=True)
else:
    ROOT = os.path.abspath('..')
import numpy as np, pandas as pd
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
warnings.filterwarnings('ignore', category=FutureWarning)
import bertopic, sentence_transformers, umap, hdbscan, gensim, underthesea
from src.clustering import core as C
CFG = C.load_config(ROOT)
if ON_KAGGLE:
    def find_file(name, root='/kaggle/input'):
        hits = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
        if not hits:
            raise FileNotFoundError(f'Không tìm thấy {name} trong {root} — gắn output của notebook 11 làm input')
        return hits[0]
    IN = {f: find_file(f) for f in ('actionable_reviews.csv', 'gd11_run_info.json')}
    RES = '/kaggle/working/rq2_topics'
else:
    RES = os.path.join(ROOT, '05_experiments', 'rq2_topics')
    IN = {f: os.path.join(RES, f) for f in ('actionable_reviews.csv', 'gd11_run_info.json')}
CACHE = os.path.join(RES, 'cache')
os.makedirs(RES, exist_ok=True)
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
VERSIONS = {m.__name__: getattr(m, '__version__', '?') for m in (bertopic, sentence_transformers, umap, gensim, underthesea)}
VERSIONS['hdbscan'] = __import__('importlib.metadata').metadata.version('hdbscan')
print('Ngày chạy:', datetime.date.today().strftime('%d/%m/%Y'), '| commit:', COMMIT)
print(VERSIONS)

## Dữ liệu

**Kiểm tra đầu vào**

Số review phải khớp `gd11_run_info.json`; mỗi review có đúng một nhãn tầng 2. Cột `split` cho biết review nào thuộc Train,
Validation, Hold-out hay focused set; mọi review đều được gom.

In [ ]:
ACT = pd.read_csv(IN['actionable_reviews.csv'], encoding='utf-8-sig')
INFO11 = json.load(open(IN['gd11_run_info.json'], encoding='utf-8'))
assert len(ACT) == INFO11['n_actionable'] and ACT['review_id'].is_unique
assert set(ACT['tier2_label']) <= set(C.TYPES)
N = len(ACT)
print(f'Review actionable: {N:,}')
print('Theo nhãn tầng 2:', ACT['tier2_label'].value_counts().to_dict())
print('Theo tập:', ACT['split'].value_counts().to_dict())

## Văn bản cho c-TF-IDF và coherence

**Tách từ**

underthesea tách từ và chuyển chữ thường; từ ghép được nối bằng "_" (ví dụ `giao_hàng`). Từ dừng (trong config, gồm tên app vì có mặt
ở gần như mọi review), token không có chữ cái và token ngắn hơn 2 ký tự bị bỏ. c-TF-IDF và C_v dùng chung các token này.

**Embedding không dùng văn bản đã lọc**

Sentence embedding tính trên văn bản gốc; văn bản đã lọc chỉ phục vụ việc mô tả cụm.

In [ ]:
SEG = C.segment(ACT['text'])
TOKENS = C.tokens_for_topics(SEG, CFG['stopwords'], CFG['vectorizer']['min_token_chars'])
DOCS = [' '.join(t) for t in TOKENS]
n_empty = sum(1 for t in TOKENS if not t)
print(f'Token / review (trung vị): {np.median([len(t) for t in TOKENS]):.0f} | review không còn token: {n_empty} ({n_empty / N:.1%})')
print('Số từ khác nhau:', len({w for t in TOKENS for w in t}))

## Sentence embedding (Q13)

**Mô hình chính**

`paraphrase-multilingual-mpnet-base-v2`, revision pin trong config. Embedding được lưu đệm trong `05_experiments/rq2_topics/cache/`
(không commit).

**So nhanh với một Vietnamese SBERT**

`keepitreal/vietnamese-sbert` (PhoBERT, nhận văn bản đã tách từ) được so với mô hình chính trên phần non-test — bỏ review thuộc
Hold-out và focused set — ở cùng một `min_topic_size` (30): C_v, tỷ lệ outlier, số cụm và từ khóa của các cụm lớn nhất. Theo Q13,
đổi mô hình phải ghi amendment trước khi khóa cấu hình BERTopic; notebook giữ mô hình chính và chỉ báo kết quả so sánh.

In [ ]:
EMB = C.embed(ACT['text'].astype(str).tolist(), ACT['review_id'], CFG['embedding'], CACHE, 'mpnet_actionable')
print('Embedding chính:', EMB.shape)

nt = ~ACT['split'].isin(['holdout', 'focused']).to_numpy()
cand_cfg = CFG['embedding_candidate']
EMB_C = C.embed([SEG[i] for i in np.where(nt)[0]], ACT['review_id'][nt], cand_cfg, CACHE, 'visbert_nontest')
docs_nt = [DOCS[i] for i in np.where(nt)[0]]
tok_nt = [TOKENS[i] for i in np.where(nt)[0]]
mts_c = cand_cfg['compare_min_topic_size']
cmp_rows, cmp_words = [], {}
for name, emb in (('mpnet (chính)', EMB[nt]), ('vietnamese-sbert', EMB_C)):
    m, t = C.fit_bertopic(docs_nt, emb, mts_c, CFG)
    cv, _ = C.coherence_cv(C.topic_words(m, CFG['selection']['top_n_words']), tok_nt)
    cmp_rows.append({'embedding': name, 'n review': int(nt.sum()), 'min_topic_size': mts_c, 'C_v': cv, **C.topic_stats(t)})
    sizes = pd.Series(t[t != -1]).value_counts()
    cmp_words[name] = [(int(k), int(v), ', '.join(C.topic_words(m, 6)[k])) for k, v in sizes.head(8).items()]
EMB_CMP = pd.DataFrame(cmp_rows)
print(EMB_CMP.round(4).to_string(index=False))
for name, ws in cmp_words.items():
    print(f'\n{name} — 8 cụm lớn nhất (id, số review, từ khóa):')
    for w in ws:
        print('  ', w)
better = EMB_CMP.loc[1, 'C_v'] > EMB_CMP.loc[0, 'C_v']
print('\nVietnamese SBERT có C_v cao hơn:', better, '— giữ mô hình chính; đổi mô hình cần amendment (Q13).' if better else '— giữ mô hình chính.')

## Chọn min_topic_size theo C_v (Q12)

**Lưới**

Mỗi giá trị {15, 20, 30, 50, 75} được fit một lần với cùng embedding, UMAP (random_state 42) và HDBSCAN. C_v tính trên 10 từ khóa đầu
của mỗi cụm (không tính outlier). Tín hiệu phụ: số cụm, tỷ lệ outlier, cụm lớn nhất và kích thước cụm.

**Luật chọn**

C_v cao nhất trong các cấu hình không suy biến; hòa thì chọn `min_topic_size` lớn hơn.

In [ ]:
GRID, FITS = C.grid(DOCS, TOKENS, EMB, CFG)
MTS = C.choose_min_topic_size(GRID)
print(); print(GRID.round(4).to_string(index=False))
print('\nmin_topic_size được chọn:', MTS)

## Giảm outlier và gộp cụm gần trùng

**Giảm outlier**

Review outlier được gán lại cho cụm gần nhất chỉ khi cosine giữa embedding của review và tâm cụm ít nhất bằng phân vị 10 của cosine
giữa các thành viên và tâm cụm của chính chúng, tức là outlier phải gần tâm ít nhất như 10% thành viên xa nhất; review xa mọi cụm vẫn
là outlier. Phân bố độ tương đồng của các outlier được in ra để thấy ngưỡng giữ lại bao nhiêu.

**Gộp cụm**

Hai cụm có tâm embedding gần trùng (cosine ≥ 0,90) được gộp theo từng cặp, cặp giống nhau nhất trước, mỗi cụm gộp tối đa một lần —
không gộp dây chuyền. Các cặp gộp được in ra để kiểm tra.

In [ ]:
MODEL, T0 = FITS[MTS]
s0 = C.topic_stats(T0)
sim = C.outlier_similarity(EMB, T0)
OUT_THR = C.outlier_threshold(EMB, T0, CFG)
print('Trước khi giảm outlier:', s0)
print(f'Ngưỡng gán lại outlier (cosine): {OUT_THR:.3f}')
if len(sim):
    print('Cosine của outlier tới tâm cụm gần nhất — phân vị 10/25/50/75/90:', np.round(np.percentile(sim, [10, 25, 50, 75, 90]), 3))
    print(f'Outlier được gán lại (cosine ≥ ngưỡng): {(sim >= OUT_THR).mean():.1%}')
T1 = C.reduce_outliers(MODEL, DOCS, T0, EMB, CFG, OUT_THR)
print('Sau khi giảm outlier:', C.topic_stats(T1))
T2, GROUPS = C.merge_duplicates(MODEL, DOCS, T1, EMB, CFG)
print('Cặp cụm được gộp (id trước khi gộp, cosine):', [(a, b, round(c, 3)) for a, b, c in GROUPS] or 'không có')
print('Sau khi gộp:', C.topic_stats(T2))
CV, CV_PER = C.coherence_cv(C.topic_words(MODEL, CFG['selection']['top_n_words']), TOKENS)
print(f'C_v của cấu hình cuối: {CV:.4f}')

## Loại chủ đạo và type purity

**Cách gắn**

Với mỗi cụm, đếm nhãn tầng 2 của các review; loại chiếm ít nhất 60% là loại chủ đạo, nếu không thì `mixed` và gắn cờ. Type purity của
cụm là tỷ lệ của loại đông nhất. Với cụm `mixed`, các loại có ít nhất 3 review hoặc ít nhất 10% cụm được ghi lại (mixed-origin, Q14)
để GĐ14 sinh requirement theo từng loại.

**Giới hạn**

Nhãn tầng 2 là nhãn dự đoán của ViSoBERT (F1 FEATURE end-to-end trên Hold-out 34,8 ± 3,6), nên loại chủ đạo và purity mang theo sai
số của bộ phân loại.

In [ ]:
DOM = C.dominant_types(T2, ACT['tier2_label'], CFG)
d = DOM[DOM['topic'] != -1]
print('Số cụm theo loại chủ đạo:', d['dominant_type'].value_counts().to_dict())
print(f'Type purity trung bình có trọng số theo kích thước: {C.type_purity_overall(DOM):.3f} | trung vị theo cụm: {d["type_purity"].median():.3f}')

## Tên cụm

**Tên theo từ khóa**

Mỗi cụm có tên tạm ghép từ 4 từ khóa c-TF-IDF đầu tiên.

**Tên bằng LLM**

Prompt `04_prompts/topic_naming/topic_naming_system_v1.txt` (hash trong registry), `gpt-4o-mini-2024-07-18`, temperature 0; đầu vào là
10 từ khóa và 5 review gần tâm cụm nhất. Chạy thử trên 3 cụm trước; chỉ đặt tên toàn bộ khi `CONFIRM_LLM_NAMING = True` sau khi đã
xem kết quả chạy thử. Tên không thay đổi thành viên của cụm. Trên Kaggle bước này được bỏ qua (API key chỉ đọc từ `.env` trên máy
local); có thể chạy lại riêng phần đặt tên trên máy local sau khi tải kết quả về.

In [ ]:
TOPICS = C.topics_table(MODEL, T2, DOM, CV_PER, N, CFG)
REP = C.representative_index(EMB, T2, CFG['naming']['n_docs'])
ITEMS = [{'topic_id': t, 'keywords': C.topic_words(MODEL, CFG['selection']['top_n_words'])[t],
          'examples': ACT['text'].iloc[idx].astype(str).str.slice(0, 300).tolist()} for t, idx in REP.items()]

RUN_PILOT, CONFIRM_LLM_NAMING = not ON_KAGGLE, False
NAMES_PATH = os.path.join(RES, 'gd12_topic_names_llm.csv')
if RUN_PILOT or CONFIRM_LLM_NAMING:
    from src.utils.llm import get_client, prices
    from src.clustering.naming import load_prompt, registered, name_topics, cost
    SYSTEM, PHASH = load_prompt('v1')
    assert registered(PHASH), 'prompt đặt tên chưa có trong 04_prompts/prompts_registry.md'
    client, SETTINGS = get_client()
    LOG = os.path.join(RES, 'gd12_topic_naming_calls.jsonl')
    if CONFIRM_LLM_NAMING and not os.path.exists(NAMES_PATH):
        NAMES = name_topics(ITEMS, SYSTEM, client, SETTINGS, LOG)
        NAMES.to_csv(NAMES_PATH, index=False)
    elif os.path.exists(NAMES_PATH):
        NAMES = pd.read_csv(NAMES_PATH)
    else:
        NAMES = name_topics(ITEMS[:3], SYSTEM, client, SETTINGS, os.path.join(RES, 'gd12_topic_naming_pilot.jsonl'))
    c = cost(NAMES, prices())
    print('Đặt tên bằng LLM:', 'toàn bộ' if len(NAMES) == len(ITEMS) else f'chạy thử {len(NAMES)} cụm', '|', c)
    print(f"Ước lượng cho {len(ITEMS)} cụm: ~${c['chi_phí_usd'] / len(NAMES) * len(ITEMS):.4f}")
    print(NAMES[['topic_id', 'name_vi', 'name_en']].to_string(index=False))
    TOPICS = TOPICS.merge(NAMES[['topic_id', 'name_vi', 'name_en']], on='topic_id', how='left')

## Tóm tắt các cụm

**Bảng**

Mỗi cụm: id, tên (LLM nếu đã đặt, nếu không là tên theo từ khóa), số review, tỷ lệ trên tổng actionable, loại chủ đạo, type purity và
từ khóa. Dòng `-1` là outlier.

**File xuất (`05_experiments/rq2_topics/`)**

`topics.csv` và `review_topic_map.csv` (review_id → topic, kèm topic trước khi giảm outlier, nhãn tầng 2 và tập) không chứa văn bản.
`gd12_topic_examples.csv` chứa review tiêu biểu của từng cụm để duyệt bằng mắt, có văn bản nên không commit. Kèm bảng lưới, bảng so sánh
embedding, các cặp cụm đã gộp và thông tin lần chạy.

In [ ]:
show = TOPICS.copy()
if 'name_vi' in show:
    show['tên'] = show['name_vi'].fillna(show['keyword_name'])
else:
    show['tên'] = show['keyword_name']
cols = ['topic_id', 'tên', 'size', 'frequency', 'dominant_type', 'type_purity', 'top_keywords']
with pd.option_context('display.max_colwidth', 70, 'display.width', 250):
    print(show[cols].assign(frequency=show['frequency'].round(4), type_purity=show['type_purity'].round(3)).to_string(index=False))
n_out = int((T2 == -1).sum())
print(f'\nSố cụm: {int((show["topic_id"] != -1).sum())} | outlier: {n_out} ({n_out / N:.1%}) | C_v: {CV:.4f} | '
      f'type purity (trọng số): {C.type_purity_overall(DOM):.3f} | cụm mixed: {int(show["flag_mixed"].sum())}')

MAP = pd.DataFrame({'review_id': ACT['review_id'], 'topic_id': T2, 'topic_before_outlier_reduction': T0,
                    'tier2_label': ACT['tier2_label'], 'split': ACT['split']})
EX = pd.DataFrame([{'topic_id': t, 'rank': r + 1, 'review_id': ACT['review_id'].iloc[i], 'tier2_label': ACT['tier2_label'].iloc[i],
                    'text': ACT['text'].iloc[i]} for t, idx in REP.items() for r, i in enumerate(idx)])
TOPICS.to_csv(os.path.join(RES, 'topics.csv'), index=False, encoding='utf-8-sig')
MAP.to_csv(os.path.join(RES, 'review_topic_map.csv'), index=False)
EX.to_csv(os.path.join(RES, 'gd12_topic_examples.csv'), index=False, encoding='utf-8-sig')
GRID.to_csv(os.path.join(RES, 'gd12_grid.csv'), index=False)
EMB_CMP.to_csv(os.path.join(RES, 'gd12_embedding_compare.csv'), index=False)
pd.DataFrame(GROUPS, columns=['topic_a', 'topic_b', 'cosine']).to_csv(os.path.join(RES, 'gd12_merge_pairs.csv'), index=False)
json.dump({'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'versions': VERSIONS, 'n_actionable': N,
           'min_topic_size': MTS, 'n_topics': int((show['topic_id'] != -1).sum()), 'n_outliers': n_out, 'c_v': CV,
           'type_purity_weighted': C.type_purity_overall(DOM), 'embedding': CFG['embedding'], 'outlier_threshold': OUT_THR,
           'n_merged_pairs': len(GROUPS),
           'llm_names': bool('name_vi' in TOPICS and TOPICS['name_vi'].notna().sum() == (TOPICS['topic_id'] != -1).sum())},
          open(os.path.join(RES, 'gd12_run_info.json'), 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if not f.startswith('cache')))

## Nhận xét

**Chưa viết**

Phần nhận xét được viết sau khi duyệt cụm bằng mắt; cấu hình chỉ được khóa sang `03_configs/frozen/` sau bước duyệt đó.